# Prompt Engineering with the OpenAI API

This notebook demonstrates four common prompting techniques:

1. **Zero-shot Prompting**
2. **Few-shot Prompting**
3. **Chain-of-Thought Prompting**
4. **System Prompting**

Each technique includes an explanation and a Python example using the OpenAI API.

## Setup

Before running the notebook, install the required packages:

```bash
pip install openai python-dotenv
```

In [5]:
# Import the required libraries.
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from the .env file.
load_dotenv()

# Retrieve the API key securely from the environment.
api_key = os.getenv("OPENAI_API_KEY")

# Stop with a clear message if the API key is missing.
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Please add it to your .env file."
    )


# Model used for all calls. Override with the OPENAI_MODEL environment variable.
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# Create the OpenAI API client.
client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


## 1. Zero-shot Prompting

### What is Zero-shot Prompting?

Zero-shot prompting asks the model to perform a task **without providing any examples**.

The model receives only an instruction and the input that it needs to process.

### When should you use it?

Zero-shot prompting works well when:

- The task is simple and clearly defined.
- The expected behavior is easy to describe.
- Examples are unnecessary.
- You want to keep the prompt short.

### Example

We will use **sentiment classification**. The model must classify a customer review as `Positive`, `Negative`, or `Neutral`.

There are no labelled examples in the prompt.

In [6]:
# Zero-shot prompt:
# The model receives an instruction but no examples.

zero_shot_prompt = """
Classify the sentiment of the following customer review.

Possible labels:
- Positive
- Negative
- Neutral

Return only the sentiment label.

Customer review:
"The delivery arrived two days early, and the product works perfectly."
"""

try:
    # Send the prompt to the OpenAI API.
    response = client.responses.create(
        model=model_name,
        input=zero_shot_prompt
    )

    # Print the model's response clearly.
    print("=== Zero-shot Response ===")
    print(response.output_text.strip())

except Exception as e:
    # Handle API errors gracefully.
    print(f"Zero-shot API request failed: {type(e).__name__}: {e}")

=== Zero-shot Response ===
Positive


## 2. Few-shot Prompting

### What is Few-shot Prompting?

Few-shot prompting provides the model with a small number of examples before
asking it to process a new input.

The examples demonstrate the expected relationship between the input and output.

### When should you use it?

Few-shot prompting is useful when:

- The task is ambiguous.
- The desired output format needs to be demonstrated.
- Examples can clarify the classification criteria.
- You want the model to follow a particular pattern.

We will use the **same sentiment classification task** as before, but this time
we will provide three labelled examples.

In [7]:
# Few-shot prompt:
# The examples demonstrate how reviews should be classified.

few_shot_prompt = """
Classify each customer review as exactly one of:
Positive, Negative, or Neutral.

Use the examples below as guidance.

Example 1:
Review: "The headphones sound fantastic and the battery lasts all day."
Sentiment: Positive

Example 2:
Review: "The package arrived damaged and the device does not turn on."
Sentiment: Negative

Example 3:
Review: "The item arrived yesterday. I have not used it yet."
Sentiment: Neutral

Now classify the following review:

Review:
"The support team answered my question quickly and solved my problem."

Return only the sentiment label.
"""

try:
    # Send the few-shot prompt to the API.
    response = client.responses.create(
        model=model_name,
        input=few_shot_prompt
    )

    print("=== Few-shot Response ===")
    print(response.output_text.strip())

except Exception as e:
    # Handle API errors gracefully.
    print(f"Few-shot API request failed: {type(e).__name__}: {e}")

=== Few-shot Response ===
Positive


## 3. Chain-of-Thought Prompting

### What is Chain-of-Thought Prompting?

Chain-of-thought prompting encourages the model to approach a problem through
multiple reasoning steps instead of immediately giving an answer.

For example, it can be useful for:

- Multi-step mathematics
- Logical reasoning
- Planning
- Problems involving several calculations

### When should you use it?

Use this approach when the task requires multiple intermediate steps.

In this example, we will use a multi-step word problem and ask the model to
"think step by step."

For production applications, it is generally better to request a concise
explanation or solution rather than requiring exposure of private internal
reasoning.

In [8]:
# Chain-of-thought style prompt:
# The problem requires multiple calculations.

cot_prompt = """
Solve the following word problem.

A bookstore has 240 books in stock.

On Monday, it sells 25% of its stock.

On Tuesday, it sells 30% of the books that remained after Monday.

On Wednesday, the bookstore receives 50 new books.

Think step by step and calculate how many books are in stock
at the end of Wednesday.

Give the final answer clearly along with a concise explanation.
"""

try:
    # Send the reasoning problem to the API.
    response = client.responses.create(
        model=model_name,
        input=cot_prompt
    )

    print("=== Chain-of-Thought Response ===")
    print(response.output_text.strip())

except Exception as e:
    # Handle API errors gracefully.
    print(f"Chain-of-thought API request failed: {type(e).__name__}: {e}")

=== Chain-of-Thought Response ===
Let's calculate the number of books in stock step by step.

1. **Initial Stock**: 
   The bookstore starts with 240 books.

2. **Monday's Sales**: 
   - It sells 25% of its stock:
     \[
     \text{Books sold on Monday} = 25\% \times 240 = \frac{25}{100} \times 240 = 60
     \]
   - Remaining stock after Monday:
     \[
     \text{Remaining stock} = 240 - 60 = 180
     \]

3. **Tuesday's Sales**: 
   - It sells 30% of the remaining stock:
     \[
     \text{Books sold on Tuesday} = 30\% \times 180 = \frac{30}{100} \times 180 = 54
     \]
   - Remaining stock after Tuesday:
     \[
     \text{Remaining stock} = 180 - 54 = 126
     \]

4. **Wednesday's New Arrivals**: 
   - The bookstore receives 50 new books:
     \[
     \text{Total stock at the end of Wednesday} = 126 + 50 = 176
     \]

Therefore, at the end of Wednesday, the bookstore has **176 books** in stock.


## 4. System Prompt

### What is a System Prompt?

A system prompt provides high-level instructions that guide the model's behavior.

It can define things such as:

- The assistant's role
- Response style
- Output format
- Rules and restrictions
- Tone of communication

### When should you use it?

System prompts are particularly useful when an application needs the model to
behave consistently across many user requests.

### Example

We will create an assistant that answers questions **only in rhyming couplets**.

The system message establishes this behavior, while the user message asks a
normal question.

In [9]:
# System prompt:
# This defines a behavioral constraint for the assistant.

system_prompt = """
You are a helpful assistant that only answers in rhyming couplets.

Every response must:
1. Contain pairs of lines.
2. Make each pair rhyme.
3. Answer the user's question clearly.
4. Avoid explanations outside the rhyming couplets.
"""

# This is the user's actual question.
user_prompt = """
Explain why Python is popular for data science.
"""

try:
    # Provide both the system instruction and user request.
    response = client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    print("=== System Prompt Response ===")
    print(response.output_text.strip())

except Exception as e:
    # Handle API errors gracefully.
    print(f"System-prompt API request failed: {type(e).__name__}: {e}")

=== System Prompt Response ===
In Python's embrace, data science takes flight,  
With libraries like Pandas that make things feel right.  

Its syntax is clear, both simple and neat,  
For beginners and pros, it can't be beat.  

From NumPy to SciPy, tools that amaze,  
They help crunch the numbers in countless ways.  

With Matplotlib’s graphs, insights come alive,  
In the world of data, Python will thrive.


## Final Summary

The four prompting techniques provide different ways of guiding a language model.

| Technique | Main Idea | Best Used When |
|---|---|---|
| **Zero-shot** | Give an instruction without examples | The task is simple and clearly defined |
| **Few-shot** | Provide labelled examples before the test input | Examples help demonstrate the desired behavior |
| **Chain-of-Thought** | Encourage multi-step reasoning | The problem requires several reasoning or calculation steps |
| **System Prompt** | Establish behavioral rules or constraints | Consistent behavior, role, style, or formatting is required |

### Key Takeaways

- **Zero-shot prompting** is simple and requires minimal prompt context.
- **Few-shot prompting** demonstrates the expected input/output pattern through examples.
- **Chain-of-thought prompting** can be useful for complex, multi-step problems.
- **System prompts** establish higher-level behavioral instructions.
- These techniques can also be combined in a single application.

For example, an application could use a system prompt to define the assistant's
role, few-shot examples to demonstrate the expected output format, and a user
prompt containing the specific task.